# Reading, PA — LVT Model (Direct AVM Hedonic Land Values)

**Reform:** 4:1 split-rate (land taxed at 4× the improvement rate), revenue-neutral  
**Levy scope:** City of Reading municipal real estate tax only  
**Land value method:** Direct AVM — `prediction_land_sqft × lot_area_sqft` from  
berks_open_avmkit hedonic_land/ensemble model (valuation date 2026-01-01)  
**Improvement value:** `max(0, hedonic_land_prediction − avm_land_value)`  
**Current tax base:** 1994 CAMA assessed values (same as model.ipynb)  
**Data sources:** Berks County GIS / CAMA_Master + berks_open_avmkit hedonic_land/ensemble AVM

## AVM hedonic land method

berks_open_avmkit trains a separate hedonic land model for each property class
(residential_sf, residential_mf, commercial, vacant). The `prediction_land_sqft`
column in those outputs is the model's direct estimate of land value per square foot.

`avm_land_value = prediction_land_sqft × lot_area_sqft`

This is more direct than the LYCD 20%-allocation approach: the model is trained
specifically to predict land value, not decomposed from a total-value prediction.
However, commercial and vacant models have sparse transaction data (COD 77–91%),
so those parcels' land values are less reliable than the residential estimates
(residential_sf: Median Ratio=1.02, COD=12.6%).

In [ ]:
import sys
import concurrent.futures
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from scipy.spatial import cKDTree

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'reading'
STATE_FIPS = '42'
COUNTY_FIPS = '011'
MODEL_TYPE = 'split_rate:4.0_avm_land'
LAND_IMPROVEMENT_RATIO = 4.0
CITY_MILLAGE = 19.217
OFFICIAL_REVENUE = 27_235_265

AVM_BASE = Path('C:/projects/berks_open_avmkit/notebooks/pipeline/data/us-pa-berks/out/models')
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Load Parcel Data

In [ ]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
if not PARCEL_PATH.exists():
    raise FileNotFoundError(
        f"Parcel cache not found at {PARCEL_PATH}.\n"
        "Run model.ipynb first (it creates the cache from berks_open_avmkit)."
    )
gdf = gpd.read_parquet(PARCEL_PATH)
print(f"Loaded {len(gdf):,} Reading parcels from cache")

for col in ['assr_land_value', 'assr_impr_value', 'assr_market_value', 'land_area_sqft']:
    gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0.0)

print(f"land_area_sqft > 0: {(gdf['land_area_sqft'] > 0).sum():,}")
print(f"land_area_sqft = 0: {(gdf['land_area_sqft'] <= 0).sum():,}")

## Section 3 — Load AVM Hedonic Land Predictions

Load `prediction_land_sqft` from the hedonic_land/ensemble model for each group.
Also load `prediction` (total hedonic value) to derive implied improvement value.
Priority: residential_sf → residential_mf → commercial → vacant.

In [ ]:
MODEL_GROUPS = ['residential_sf', 'residential_mf', 'commercial', 'vacant']

frames = []
for i, mg in enumerate(MODEL_GROUPS):
    path = AVM_BASE / mg / 'hedonic_land' / 'ensemble' / 'pred_universe.parquet'
    if path.exists():
        df = pd.read_parquet(path, columns=['key', 'prediction_land_sqft', 'prediction'])
        df['_priority'] = i
        frames.append(df)
        print(f"  {mg} hedonic_land: {len(df):,} county parcels")
    else:
        print(f"  {mg}: hedonic_land not found at {path}")

if not frames:
    raise FileNotFoundError("No hedonic_land pred_universe files found under AVM_BASE.")

hl = (
    pd.concat(frames)
    .sort_values('_priority')
    .drop_duplicates('key', keep='first')
    .drop(columns='_priority')
    .rename(columns={
        'prediction_land_sqft': 'hl_land_psf',
        'prediction': 'hl_prediction',
    })
)
print(f"\nHedonic land predictions (county-wide): {len(hl):,} unique parcels")

gdf = gdf.merge(hl, on='key', how='left')
n_covered = gdf['hl_land_psf'].notna().sum()
print(f"Reading parcels with hedonic land PSF: {n_covered:,} / {len(gdf):,} ({n_covered/len(gdf):.1%})")
print(f"Uncovered (will use KNN fallback):     {gdf['hl_land_psf'].isna().sum():,}")

print(f"\nHedonic land PSF stats (Reading, covered):")
print(gdf['hl_land_psf'].describe().round(2))

## Section 4 — Classify Parcels

In [ ]:
gdf['full_exmp'] = (gdf['category_code'] == 'E').astype(int)

def classify_parcel(row):
    cat = str(row.get('category_code', ''))
    luc = str(row.get('luc', ''))
    if cat == 'E':
        return 'Exempt'
    elif cat == 'R':
        if luc.endswith('R') or luc in ['162', '163']:
            return 'Townhome / Rowhouse'
        elif luc[:2] in ['12', '13']:
            return 'Small Multi-Family (2-4 units)'
        elif luc in ['149', '153']:
            return 'Other Residential'
        else:
            return 'Single Family Residential'
    elif cat == 'A':
        return 'Large Multi-Family (5+ units)'
    elif cat in ['C', 'I']:
        luc_prefix = luc[:2]
        if luc_prefix == '42':
            return 'Large Multi-Family (5+ units)'
        elif luc_prefix in ['23', '24']:
            return 'Small Multi-Family (2-4 units)'
        elif cat == 'I' or luc_prefix in ['51', '52', '58', '59']:
            return 'Industrial'
        else:
            return 'Commercial'
    elif cat == 'F':
        return 'Agricultural'
    elif cat in ['UE', 'UT']:
        return 'Other'
    else:
        return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf.apply(classify_parcel, axis=1)
gdf.loc[gdf['assr_impr_value'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

print("Property category distribution:")
print(gdf['PROPERTY_CATEGORY'].value_counts())

## Section 5 — Current Tax (1994 CAMA)

In [ ]:
gdf['millage_rate'] = CITY_MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='assr_market_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
print(f"Modeled gross levy:  ${current_revenue:,.0f}")
print(f"Official target:     ${OFFICIAL_REVENUE:,.0f}")
print(f"Gap:                 {gap_pct:+.2f}%")

## Section 6 — AVM Land Values

`avm_land_value = hl_land_psf × land_area_sqft`  
KNN fallback (K=5 nearest covered parcels) for the ~79 uncovered parcels.  
For improved parcels, land is capped at the hedonic total prediction.

In [ ]:
# Project to EPSG:3857 once for all spatial operations in this cell
gdf_m = gdf.to_crs('EPSG:3857')
coords_all = np.column_stack([gdf_m.geometry.centroid.x, gdf_m.geometry.centroid.y])

has_hl   = gdf['hl_land_psf'].notna() & (gdf['hl_land_psf'] > 0)
has_area = gdf['land_area_sqft'] > 0

# KNN-impute lot area for parcels with zero/null area
land_area = gdf['land_area_sqft'].values.copy()
no_area_mask = land_area <= 0
if no_area_mask.sum() > 0:
    has_area_idx = np.where(has_area.values)[0]
    tree_area = cKDTree(coords_all[has_area_idx])
    _, area_nn = tree_area.query(coords_all[no_area_mask], k=min(5, len(has_area_idx)))
    land_area[no_area_mask] = np.median(
        gdf['land_area_sqft'].values[has_area_idx][area_nn],
        axis=1 if area_nn.ndim == 2 else None
    )
    print(f"KNN-imputed lot area for {no_area_mask.sum():,} parcels")

gdf['avm_land_value'] = np.where(
    has_hl & (land_area > 0),
    (gdf['hl_land_psf'] * land_area).clip(lower=0),
    np.nan,
)

# KNN fallback for uncovered parcels
n_missing = gdf['avm_land_value'].isna().sum()
if n_missing > 0:
    covered_mask = gdf['avm_land_value'].notna()
    covered_idx  = np.where(covered_mask.values)[0]
    missing_idx  = np.where(~covered_mask.values)[0]
    tree_cov = cKDTree(coords_all[covered_idx])
    _, nn = tree_cov.query(coords_all[missing_idx], k=min(5, len(covered_idx)))
    covered_vals = gdf['avm_land_value'].values[covered_idx]
    imputed = np.median(covered_vals[nn], axis=1) if nn.ndim == 2 else covered_vals[nn]
    gdf.loc[~covered_mask, 'avm_land_value'] = imputed
    print(f"KNN-imputed avm_land_value for {n_missing:,} uncovered parcels")

# Cap at hedonic total for improved parcels (land cannot exceed total value)
is_improved = gdf['assr_impr_value'] > 0
hl_total = gdf['hl_prediction'].fillna(gdf['avm_land_value'])
cap_mask = is_improved & (gdf['avm_land_value'] > hl_total)
n_capped = cap_mask.sum()
gdf.loc[cap_mask, 'avm_land_value'] = hl_total[cap_mask]
if n_capped > 0:
    print(f"Capped avm_land_value at hedonic total for {n_capped:,} improved parcels")

# Implied improvement = hedonic total - AVM land
gdf['avm_impr_value'] = (
    hl_total - gdf['avm_land_value']
).clip(lower=0)

taxable_mask = gdf['full_exmp'] == 0
print(f"\nAVM land base   (taxable parcels): ${gdf.loc[taxable_mask, 'avm_land_value'].sum()/1e6:.1f}M")
print(f"AVM impr base   (taxable parcels): ${gdf.loc[taxable_mask, 'avm_impr_value'].sum()/1e6:.1f}M")
print(f"CAMA land base  (taxable parcels): ${gdf.loc[taxable_mask, 'assr_land_value'].sum()/1e6:.1f}M")
print(f"Ratio AVM/CAMA land:               "
      f"{gdf.loc[taxable_mask,'avm_land_value'].sum() / gdf.loc[taxable_mask,'assr_land_value'].sum():.2f}x")

print(f"\nAVM land value stats (taxable):")
print(gdf.loc[taxable_mask, 'avm_land_value'].describe().round(0))

## Section 7 — 4:1 Split-Rate Model (AVM Hedonic Land Base)

In [ ]:
taxable = gdf[gdf['full_exmp'] == 0].copy()
taxable['taxable_land_value'] = taxable['avm_land_value']
taxable['taxable_improvement_value'] = taxable['avm_impr_value']

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

exempt = gdf[gdf['full_exmp'] == 1].copy()
for col in ['new_tax', 'tax_change', 'tax_change_pct', 'taxable_land_value', 'taxable_improvement_value']:
    exempt[col] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f"Land millage:        {land_millage:.4f} mills")
print(f"Improvement millage: {improvement_millage:.4f} mills")
print(f"(Prior flat rate:    {CITY_MILLAGE:.3f} mills)")
print(f"Revenue check:       ${new_revenue:,.0f}")
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title="Reading, PA — 4:1 Split-Rate Tax Impact (AVM Hedonic Land Values)")

## Section 8 — Exploration Charts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
cat_med = (
    gdf[gdf['PROPERTY_CATEGORY'] != 'Exempt']
    .groupby('PROPERTY_CATEGORY')['tax_change_pct']
    .median()
    .sort_values()
)
colors = ['#d62728' if v > 0 else '#2ca02c' for v in cat_med.values]
cat_med.plot.barh(ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Reading, PA — Median Tax Change % by Category (4:1 Split-Rate, AVM Hedonic Land Values)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview_avm_land.png', dpi=150)
plt.close()
print("Saved category_preview_avm_land.png")

## Section 9 — Census Join + Export

In [ ]:
_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
from lvt.viz import create_city_report

out_df = save_standard_export(
    df=gdf,
    city=f'{CITY_NAME}_avm_land',
    output_path=f'../../analysis/data/{CITY_NAME}_avm_land.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

create_city_report(out_df, f'{CITY_NAME}_avm_land', show=False)
print("Done.")

## Validation Summary

| Check | Result |
|---|---|
| Revenue match | Same current tax base as model.ipynb (1994 CAMA, 19.217 mills) |
| Hedonic land coverage | 99.7% direct; 79 parcels KNN-imputed |
| Improvement base | ~$0.4M (see note below) |

**Hedonic land model behavior — effectively a pure land tax:**  
For ~48% of improved parcels, `hl_land_psf × land_area_sqft > hl_prediction`, triggering
the cap (`avm_land_value = hl_prediction`, `avm_impr_value = 0`). The remaining parcels
contribute only ~$0.4M in improvement base across 26K taxable parcels (~$16/parcel average).
In practice this model collapses the 4:1 split-rate into a near-pure land value tax.

**Why this happens:** The hedonic_land model is trained on land-only (vacant) sales and
extrapolates PSF to all parcels. In a dense urban city like Reading, lot sizes are small
but the PSF extrapolation can exceed total property value for many parcels, especially
for rowhouses with small footprints but relatively high total values. The cap prevents
land > total, but it also zeroes out the improvement base for those parcels.

**Interpretation:** This model is most useful as a sensitivity test showing what would
happen if nearly all property value were treated as land value. Compare with model_lycd.ipynb
(LYCD 20% allocation, moderate land fraction) and model.ipynb (1994 CAMA splits) to see
how sensitive results are to the land/improvement split assumption.

**Commercial and vacant quality caveats:**  
- residential_mf hedonic_land model was not found; those 50 parcels fall through to commercial  
- commercial and vacant hedonic_land models have sparse transaction data (COD expected 77–91%)  
- Residential results (22,613 parcels) are most reliable